# TriFlow scoring for decoding-design-bias

TriFlow ([Zhou lab preprint, Nov 2025](https://www.biorxiv.org/content/10.64898/2025.11.30.691458v1.full)) is a discrete flow-matching inverse folding model. It generates sequences rather than computing likelihoods natively, but at each sampling step it emits a per-position categorical distribution over the 21-AA alphabet (`unmasked_probs`, shape `(1, L, 21)`). We score the native sequence by reading its log-probability from that terminal-step distribution - analogous to how PiFold / ProteinMPNN-unconditional are scored.

**Scoring convention:**
- `triflow_score` = mean over residues of log P(native_i | structure) at the flow's terminal timestep
- Directly comparable to: **PiFold**, **ProteinMPNN (unconditional)** - same axis (structure-conditioned, parallel, no sequence context)
- Caveat: it's a flow-matching posterior marginal, not a normalizing-flow log-likelihood. Same asterisk as PiFold's "pseudo-log-likelihood."

**Config:** AFDB-trained weights (`afdb_weights.pt`) - matches the AF-predicted structures in our dataset. CFG off. Temperature set to 1.0 (affects sampling, not the probability distribution).

**Runtime on A100:** ~3-5s per protein × 7843 ≈ 7-11 hours total, resume-safe.

## 1. Setup

In [ ]:
!nvidia-smi -L

In [ ]:
# TriFlow depends on OpenFold-style helpers. environment.yaml pins torch==2.6.0 but newer
# versions also work with the model code. Install the explicit deps we need.
!pip install -q biopython==1.85 pytorch-lightning==2.5.1.post0 deepspeed==0.16.7 ml-collections einops dm-tree requests tqdm

In [ ]:
import os

TRIFLOW_URL = 'https://github.com/jzhoulab/TriFlow.git'
TRIFLOW_DIR = '/content/TriFlow'

BIAS_URL = 'https://github.com/LBDillon/decoding-design-bias.git'
BIAS_DIR = '/content/decoding-design-bias'
DATASET  = '/content/main_plus_r2_r3_scored_filterC_v3.csv'

if not os.path.exists(TRIFLOW_DIR):
    !git clone --depth 1 {TRIFLOW_URL} {TRIFLOW_DIR}
if not os.path.exists(BIAS_DIR):
    !git clone --depth 1 {BIAS_URL} {BIAS_DIR}

assert os.path.exists(f'{TRIFLOW_DIR}/weights/afdb_dataset/afdb_weights.pt'), 'weights missing'
assert os.path.exists(DATASET), DATASET

# Make TriFlow importable as a package
import sys
sys.path.insert(0, TRIFLOW_DIR)
os.chdir(TRIFLOW_DIR)  # the model expects to find weights/ relatively

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT_DIR = '/content/drive/MyDrive/decoding-design-bias/outputs'
OUTPUT = f'{DRIVE_OUT_DIR}/triflow_scores.csv'
PDB_CACHE = '/content/pdbs'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
os.makedirs(PDB_CACHE, exist_ok=True)
print('output ->', OUTPUT)

## 2. Load the TriFold predictor

First call triggers `torch.compile` warm-up (1-2 min). Subsequent forward passes are fast.

In [ ]:
import torch
import warnings; warnings.filterwarnings('ignore', category=UserWarning)

assert torch.cuda.is_available(), 'Use a GPU runtime.'

from sample import TriFoldPredictor  # top-level script in the repo

predictor = TriFoldPredictor(
    ckpt_path=f'{TRIFLOW_DIR}/weights/afdb_dataset/afdb_weights.pt',
    device='cuda:0',
)
print('ready')

## 3. PDB fetcher + in-process scoring wrapper

The wrapper bypasses `predict()` (which writes FASTA/PDB files to disk) and calls the underlying flow sampler directly, reading `unmasked_probs` from the final step.

In [ ]:
import requests

AF_URL_TEMPLATES = [
    'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v6.pdb',
    'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v4.pdb',
    'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v3.pdb',
]

def fetch_pdb(entry, cache_dir=PDB_CACHE):
    for t in AF_URL_TEMPLATES:
        url = t.format(uid=entry)
        version = url.rsplit('model_', 1)[-1].replace('.pdb', '')
        local = os.path.join(cache_dir, f'AF-{entry}-F1-model_{version}.pdb')
        if os.path.exists(local):
            return local
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                with open(local, 'wb') as fh:
                    fh.write(r.content)
                return local
        except Exception:
            continue
    return None

In [ ]:
#@title PDB-MODE (R3.3) - score on experimental PDB chains instead of AlphaFold
#@markdown Toggle ON to score the experimental-structure subset. First upload
#@markdown **pdb_scoring_inputs.csv** and unzip **pdb_chain_structs.zip** to
#@markdown `/content/`. Run this cell AFTER the config/fetch_pdb cell and BEFORE
#@markdown the validation/scoring cells. Leave OFF to use AlphaFold (default).
PDB_MODE = False  #@param {type:"boolean"}
if PDB_MODE:
    import os, pandas as pd
    DATASET = "/content/pdb_scoring_inputs.csv"   # Entry, pdb_id, pdb_chain(=A), sequence(=chain), chain_pdb_path
    assert os.path.exists(DATASET), "Upload pdb_scoring_inputs.csv to /content/"
    _CHAINDIR = "/content/pdb_chain_structs"
    _pdb_df = pd.read_csv(DATASET)
    _pmap = {r.Entry: os.path.join(_CHAINDIR, os.path.basename(str(r.chain_pdb_path)))
             for r in _pdb_df.itertuples()}
    def fetch_pdb(entry, *args, **kwargs):   # override: local single-chain experimental PDB
        p = _pmap.get(entry)
        return p if (p and os.path.exists(p)) else None
    # write/resume from a SEPARATE file so the AlphaFold run's checkpoint isn't
    # reused (otherwise resume sees the AF rows as 'already scored' -> remaining 0)
    try:
        OUTPUT = os.path.splitext(OUTPUT)[0] + "_pdb.csv"
        print("OUTPUT ->", OUTPUT)
    except NameError:
        print("WARNING: OUTPUT not defined yet - run the config cell ABOVE this one first.")
    print(f"PDB-MODE ON - {len(_pmap)} experimental-structure inputs; DATASET -> {DATASET}")
    print("'sequence' is the resolved PDB chain; structures are single-chain (chain 'A').")
    print("Scores cover the resolved region; compare to AF2 scores per-residue (R3.3).")
else:
    print("PDB-MODE OFF - using AlphaFold structures (default).")


In [ ]:
from triflow.utils.rigid_utils import Rigid
from triflow.utils.tensor_utils import tensor_tree_map
from triflow.utils.loss import scale_trans

@torch.no_grad()
def score_protein(pdb_path, temp=1.0):
    """Returns dict: triflow_score (mean log P_native), triflow_sum, length."""
    data, seq_prior = predictor.process_pdb(pdb_path=pdb_path)
    data = tensor_tree_map(lambda x: x.clone().to(predictor.device), data)
    seq_prior = seq_prior.clone().to(predictor.device)

    rigid_frames = (
        Rigid.from_tensor_4x4(data['backbone_rigid_tensor'][..., -1])
        .to_tensor_7().to(predictor.device)
    )
    rigid_frames = scale_trans(rigid_frames, 0.1)
    data['noise_label'] = torch.tensor([0.], device=predictor.device)[None]
    torch.backends.cuda.matmul.allow_tf32 = True

    _, _, unmasked_probs = predictor.interpolant.aa_sample(
        data, predictor.model, rigid_frames,
        aa_init=seq_prior, temp=temp,
        omit_AA=None, tied_weights=False,
        sample_priority=False, run_cfg=False, sample_purity=False,
    )
    # unmasked_probs: (1, L, 21)
    probs = unmasked_probs[0]  # (L, 21)
    native = torch.argmax(data['target_feat'][..., -1], dim=-1)[0]  # (L,)
    assert probs.shape[0] == native.shape[0], (probs.shape, native.shape)
    native_probs = probs[torch.arange(native.shape[0]), native].clamp(min=1e-30)
    log_probs = torch.log(native_probs)
    return {
        'triflow_score': log_probs.mean().item(),
        'triflow_sum':   log_probs.sum().item(),
        'length':        int(native.shape[0]),
    }

## 4. Sanity check on 3 proteins

In [ ]:
import csv, time

with open(DATASET) as fh:
    rows = list(csv.DictReader(fh))
print('dataset:', len(rows), 'rows')

for r in rows[:3]:
    entry = r['Entry']
    t0 = time.time(); pdb = fetch_pdb(entry); t_dl = time.time() - t0
    if pdb is None:
        print(entry, 'no PDB'); continue
    t0 = time.time(); res = score_protein(pdb); t_sc = time.time() - t0
    print(f'{entry} L={res["length"]} score={res["triflow_score"]:.4f}  dl={t_dl:.1f}s score_t={t_sc:.1f}s')

## 5. Full run with resume

In [ ]:
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore', category=UserWarning)

already = set()
if os.path.exists(OUTPUT):
    with open(OUTPUT) as fh:
        for row in csv.DictReader(fh):
            already.add(row['Entry'])
    print('resuming, already scored:', len(already))

todo = sorted([r for r in rows if r['Entry'] not in already],
              key=lambda r: int(r.get('sequence_length') or len(r.get('sequence',''))))
print('remaining:', len(todo))

open_mode = 'a' if already else 'w'
with open(OUTPUT, open_mode, newline='') as out:
    w = csv.writer(out)
    if open_mode == 'w':
        w.writerow([
            'Entry', 'species', 'domain',
            'triflow_score', 'triflow_sum',
            'scored_length', 'dataset_length',
        ])

    counts = {'ok': 0, 'missing_pdb': 0, 'error': 0}
    t_start = time.time()
    for r in tqdm(todo, desc='triflow'):
        entry = r['Entry']
        pdb = fetch_pdb(entry)
        if pdb is None:
            counts['missing_pdb'] += 1; continue
        try:
            res = score_protein(pdb)
        except Exception as exc:
            print(f'[{entry}] {exc}')
            counts['error'] += 1; continue
        w.writerow([
            entry, r.get('species',''), r.get('domain',''),
            f"{res['triflow_score']:.6f}", f"{res['triflow_sum']:.6f}",
            res['length'], len(r.get('sequence','')),
        ])
        out.flush()
        counts['ok'] += 1

print('done in', round(time.time() - t_start, 1), 's', counts)

## 6. Quick look

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT)
print(df.shape)
df['triflow_score'].describe()